# CIFAR-10 ResNet18 GPU Transfer (Google Colab)

**Runtime → Change runtime type → GPU (T4 / L4 / A100).**

Runs the repo two-stage transfer pipeline on CUDA, then lets you download `model_best.pth` for local rigorous eval.

Expected after a full run: roughly **~90%+** locked-test accuracy (vs ~81.8% ImprovedCNN+TTA on CPU).

## 1) Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"
print("torch", torch.__version__)
print("device", torch.cuda.get_device_name(0))

## 2) Clone repo + install package

Uses the public GitHub repo. Change `REPO_URL` / `BRANCH` if needed.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/koenchill/cifar_data_classification_cnn.git"
BRANCH = "dev"  # or main / your feature branch with transfer GPU CLI
ROOT = Path("/content/cifar_data_classification_cnn")

if not ROOT.exists():
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {ROOT}
else:
    print("repo already present", ROOT)

%cd {ROOT}
!pip -q install -e .
print("cwd", Path.cwd())

## 3) Optional: short smoke (2+3 epochs)

Use this first to verify the pipeline. Comment it out for the full run.

In [ ]:
# Smoke — ~a few minutes on T4
!python scripts/train_transfer_live.py --device cuda --stage1-epochs 2 --stage2-epochs 3

## 4) Full GPU transfer (recommended)

Uses `transfer_stage1_gpu.yaml` + `transfer_stage2_gpu.yaml` automatically when `--device cuda`.

In [ ]:
# Full run — tens of minutes on T4 (depends on early stop)
!python scripts/train_transfer_live.py --device cuda

## 5) Locked-test rigorous eval (+ TTA)

In [ ]:
!python scripts/evaluate_rigorous.py --live --model ResNet18CIFAR \
  --normalize imagenet \
  --checkpoint artifacts/transfer_live/model_best.pth \
  --out-dir artifacts/transfer_rigorous \
  --tta --skip-robustness

import json
from pathlib import Path
summary = json.loads(Path("artifacts/transfer_rigorous/summary.json").read_text())
print(
    f"accuracy={summary['accuracy']:.4f} "
    f"macro_P={summary['macro_precision']:.4f} "
    f"macro_R={summary['macro_recall']:.4f} "
    f"macro_F1={summary['macro_f1']:.4f} "
    f"roc_auc={summary['roc_auc_macro']}"
)

## 6) Download artifacts to your machine

In [ ]:
from google.colab import files
from pathlib import Path

for path in [
    "artifacts/transfer_live/model_best.pth",
    "artifacts/transfer_live/run_record.json",
    "artifacts/transfer_rigorous/summary.json",
]:
    p = Path(path)
    if p.is_file():
        files.download(str(p))
    else:
        print("missing", p)

## Local follow-up

After downloading `model_best.pth` into your laptop repo:

```powershell
.\.venv\Scripts\python.exe scripts/evaluate_rigorous.py --live --model ResNet18CIFAR `
  --normalize imagenet --checkpoint artifacts/transfer_live/model_best.pth `
  --out-dir artifacts/transfer_rigorous --tta
```

**Note:** This Colab notebook needs the branch that contains `--device cuda` and `transfer_stage*_gpu.yaml` (push your local branch to GitHub first if those commits are not on `dev` yet).